In [ ]:
# requirements: numpy, pandas, pyarrow, tqdm, glob
import os, re, json, math, glob
import numpy as np
import pandas as pd
import warnings
from tqdm import tqdm
import pyarrow.parquet as pq

DATA_ROOT = "./data"
TRAIN_ROOT = os.path.join(DATA_ROOT, "train")
TEST_ROOT = os.path.join(DATA_ROOT, "test")
OUT_ROOT = "./processed"
os.makedirs(OUT_ROOT, exist_ok=True)

def debug_nan_stats(x, label):
    total = x.size
    nans = np.isnan(x).sum()
    print(f"{label}: shape={x.shape}, NaNs={nans} ({100*nans/total:.2f}%)")

# ------- IO helpers -------
def load_parquet_array(path, reshape=None, dtype=None):
    tbl = pq.read_table(path)
    df = tbl.to_pandas()
    arr = df.to_numpy()
    if dtype is not None:
        arr = arr.astype(dtype, copy=False)
    if reshape is not None:
        arr = arr.reshape(reshape)
    return arr

def fill_nan_map(arr, fill_value=None, per_col=False):
    a = arr.copy()
    if np.issubdtype(a.dtype, np.floating):
        if fill_value is not None:
            m = np.isnan(a)
            if m.any():
                a[m] = fill_value
        elif per_col:
            # per-column median for 2D maps
            for j in range(a.shape[1]):
                col = a[:, j]
                med = np.nanmedian(col)
                col[np.isnan(col)] = med
                a[:, j] = col
        else:
            med = np.nanmedian(a)
            a[np.isnan(a)] = med
    return a

def sanitize_flat(flat_map, eps=1e-8):
    f = flat_map.astype(np.float32, copy=True)
    bad = ~np.isfinite(f) | (np.abs(f) < eps)
    if bad.any():
        f[bad] = 1.0
    return f

def crop_center_rows(arr, rows=16):  # for AIRS: T x 32 x 356 -> T x 16 x 356
    H = arr.shape[1]
    s = (H - rows) // 2
    return arr[:, s:s+rows, :]

def crop_center_2d(arr, h=16, w=16):  # for FGS: T x 32 x 32 -> T x 16 x 16
    H, W = arr.shape[1], arr.shape[2]
    sh = (H - h) // 2
    sw = (W - w) // 2
    return arr[:, sh:sh+h, sw:sw+w]

def cds_pairwise(x):  # T x ... -> (T//2) x ...
    T = x.shape[0]
    T2 = (T // 2) * 2
    a = x[:T2:2]
    b = x[1:T2:2]
    return b - a

def make_index_bins(n_frames, factor):
    # contiguous, non-overlapping bins of size=factor; drop tail
    m = n_frames // factor
    bins = [(i*factor, (i+1)*factor) for i in range(m)]
    return bins

def reduce_bins_safe(arr_T_any, bins, reducer="median"):
    out = []
    last_valid = None
    for s, e in bins:
        chunk = arr_T_any[s:e]
        valid_frames = np.isfinite(chunk).any(axis=tuple(range(1, chunk.ndim)))
        
        if valid_frames.any():
            sub = chunk[valid_frames]
            # Suppress the specific RuntimeWarning about all-NaN slices
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                if reducer == "median":
                    red = np.nanmedian(sub, axis=0)
                else:
                    red = np.nanmean(sub, axis=0)
            red = np.nan_to_num(red, nan=0.0)
        else:
            if last_valid is not None:
                red = last_valid
            else:
                red = np.zeros_like(arr_T_any[0])
        
        out.append(red.astype(np.float32, copy=False))
        last_valid = out[-1]
    return np.stack(out, axis=0)

def align_fgs_to_airs_bins_safe(fgs_times, airs_cds_times, bins, arr_T_any, reducer="median"):
    out = []
    last_valid = None
    t = fgs_times
    for s, e in bins:
        t0, t1 = airs_cds_times[s], airs_cds_times[e-1]
        idx = np.nonzero((t >= t0) & (t <= t1))[0]
        if idx.size == 0:
            j = np.searchsorted(t, 0.5*(t0+t1))
            j = int(np.clip(j, 0, len(t)-1))
            chunk = arr_T_any[j:j+1]
        else:
            chunk = arr_T_any[idx]
            
        valid_frames = np.isfinite(chunk).any(axis=tuple(range(1, chunk.ndim)))
        if valid_frames.any():
            sub = chunk[valid_frames]
            # Suppress the specific RuntimeWarning about all-NaN slices
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                if reducer == "median":
                    red = np.nanmedian(sub, axis=0)
                else:
                    red = np.nanmean(sub, axis=0)
            red = np.nan_to_num(red, nan=0.0)
        else:
            red = last_valid if last_valid is not None else np.zeros_like(arr_T_any[0])
            
        out.append(red.astype(np.float32, copy=False))
        last_valid = out[-1]
    return np.stack(out, axis=0)

axis_info_path = os.path.join(DATA_ROOT, "axis_info.parquet")
axis_df = pq.read_table(axis_info_path).to_pandas()

t_air_raw = axis_df["AIRS-CH0-axis0-h"].to_numpy()
t_fgs_raw = axis_df["FGS1-axis0-h"].to_numpy()

valid_air_mask = np.isfinite(t_air_raw)
valid_fgs_mask = np.isfinite(t_fgs_raw)

t_air_raw_clean = t_air_raw[valid_air_mask]
t_fgs_raw_clean = t_fgs_raw[valid_fgs_mask]

print(f"Found {len(t_air_raw_clean)} valid AIRS timestamps, {len(t_fgs_raw_clean)} valid FGS timestamps")


def make_cds_times(t_raw):
    n = len(t_raw)
    n2 = (n // 2) * 2  # ensure even
    if n2 < 2:
        return np.array([])
    return 0.5*(t_raw[1:n2:2] + t_raw[:n2:2])

t_air_cds = make_cds_times(t_air_raw_clean)
t_fgs_cds = make_cds_times(t_fgs_raw_clean)

print(f"Generated {len(t_air_cds)} AIRS CDS times, {len(t_fgs_cds)} FGS CDS times")

adc_info = pd.read_csv(os.path.join(DATA_ROOT, "adc_info.csv"))
def get_gain_offset(instrument):
    # instrument in {"AIRS-CH0","FGS1"}
    g = float(adc_info.loc[0, f"{instrument}_adc_gain"])
    o = float(adc_info.loc[0, f"{instrument}_adc_offset"])
    return g, o

GAIN_AIRS, OFF_AIRS = get_gain_offset("AIRS-CH0")
GAIN_FGS , OFF_FGS  = get_gain_offset("FGS1")


def process_dataset(ROOT, tag):
    planet_dirs = sorted([d for d in os.listdir(ROOT) if re.fullmatch(r"\d+", d)])
    print(f"Found {len(planet_dirs)} planets under {ROOT}")

    airs_records = []
    fgs_records = []

    for pid in tqdm(planet_dirs, desc=f"Processing {tag}"):
        pdir = os.path.join(ROOT, pid)

        airs_sig_paths = sorted(glob.glob(os.path.join(pdir, "AIRS-CH0_signal_*.parquet")))
        fgs_sig_paths  = sorted(glob.glob(os.path.join(pdir, "FGS1_signal_*.parquet")))


        def visit_idx(path): 
            m = re.search(r"_(\d+)\.parquet$", os.path.basename(path))
            return int(m.group(1)) if m else -1
        airs_visits = sorted([(visit_idx(p), p) for p in airs_sig_paths])
        fgs_visits  = sorted([(visit_idx(p), p) for p in fgs_sig_paths ])


        common_idxs = sorted(set(i for i,_ in airs_visits) & set(i for i,_ in fgs_visits))
        for vi in common_idxs:
            # paths
            air_sig = dict(airs_visits)[vi]
            fgs_sig = dict(fgs_visits)[vi]

            air_cal = os.path.join(pdir, f"AIRS-CH0_calibration_{vi}")
            fgs_cal = os.path.join(pdir, f"FGS1_calibration_{vi}")

            # load signals
            x_air = load_parquet_array(air_sig, reshape=(-1, 32, 356), dtype=np.float32)
            x_fgs = load_parquet_array(fgs_sig, reshape=(-1, 32,  32), dtype=np.float32)

            # CDS
            x_air = cds_pairwise(x_air)  # -> T_air_cds x 32 x 356
            x_fgs = cds_pairwise(x_fgs)  # -> T_fgs_cds x 32 x 32

            # calibration maps
            air_dead = load_parquet_array(os.path.join(air_cal, "dead.parquet"), reshape=(32,356))
            air_flat_raw = load_parquet_array(os.path.join(air_cal, "flat.parquet"), reshape=(32,356)).astype(np.float32)
            air_flat = sanitize_flat(fill_nan_map(air_flat_raw, fill_value=1.0))
            air_dark = fill_nan_map(load_parquet_array(os.path.join(air_cal, "dark.parquet"), reshape=(32,356)).astype(np.float32), per_col=False)

            fgs_dead = load_parquet_array(os.path.join(fgs_cal, "dead.parquet"), reshape=(32,32))
            fgs_flat_raw = load_parquet_array(os.path.join(fgs_cal, "flat.parquet"), reshape=(32,32)).astype(np.float32)
            fgs_flat = sanitize_flat(fill_nan_map(fgs_flat_raw, fill_value=1.0))
            fgs_dark = fill_nan_map(load_parquet_array(os.path.join(fgs_cal, "dark.parquet"), reshape=(32,32)).astype(np.float32), per_col=False)
            
            # apply gain (offset cancels in CDS)
            x_air = x_air * GAIN_AIRS
            x_fgs = x_fgs * GAIN_FGS

            # dark/flat
            x_air = (x_air - air_dark[None, ...]) / air_flat[None, ...]
            x_fgs = (x_fgs - fgs_dark[None, ...]) / fgs_flat[None, ...]

            # dead mask -> NaN
            x_air[:, air_dead] = np.nan
            x_fgs[:, fgs_dead] = np.nan
        
            # spatial crop
            x_air = crop_center_rows(x_air, rows=16)   # T x 16 x 356
            x_fgs = crop_center_2d(x_fgs, h=16, w=16)  # T x 16 x 16

            # temporal bins on AIRS-CDS indices
            T_air = x_air.shape[0]
            T_fgs = x_fgs.shape[0]
            
            # Safely slice available timestamps
            air_times = t_air_cds[:min(T_air, len(t_air_cds))]
            fgs_times = t_fgs_cds[:min(T_fgs, len(t_fgs_cds))]

            # Ensure we have enough timestamps for meaningful processing
            if len(air_times) < T_air * 0.9:  # at least 90% coverage
                print(f"Warning: Planet {pid} visit {vi} - insufficient AIRS timestamps")
            if len(fgs_times) < T_fgs * 0.9:
                print(f"Warning: Planet {pid} visit {vi} - insufficient FGS timestamps")

            bins10 = make_index_bins(len(air_times), 10)   # ~562 bins
            bins40 = make_index_bins(len(air_times), 40)   # ~140 bins

            # reduce AIRS in its own index bins (NaN-safe)
            air_ds10 = reduce_bins_safe(x_air[:len(air_times)], bins10, reducer="median")
            air_ds40 = reduce_bins_safe(x_air[:len(air_times)], bins40, reducer="median")

            # align FGS to AIRS bins by time (NaN-safe)
            fgs_ds120 = align_fgs_to_airs_bins_safe(fgs_times, air_times, bins10, x_fgs[:len(fgs_times)], reducer="median")
            fgs_ds480 = align_fgs_to_airs_bins_safe(fgs_times, air_times, bins40, x_fgs[:len(fgs_times)], reducer="median")

            # sanity check: temporal alignment
            assert air_ds10.shape[0] == fgs_ds120.shape[0], f"Temporal mismatch: {air_ds10.shape[0]} vs {fgs_ds120.shape[0]}"
            assert air_ds40.shape[0] == fgs_ds480.shape[0], f"Temporal mismatch: {air_ds40.shape[0]} vs {fgs_ds480.shape[0]}"

            # records (store both resolutions per sample)
            airs_records.append({
                "planet_id": int(pid),
                "visit": int(vi),
                "ds10": air_ds10.astype(np.float32),
                "ds40": air_ds40.astype(np.float32),
            })
            fgs_records.append({
                "planet_id": int(pid),
                "visit": int(vi),
                "ds120": fgs_ds120.astype(np.float32),
                "ds480": fgs_ds480.astype(np.float32),
            })

    return airs_records, fgs_records

# ------- shard into n_shards .npy files -------
def save_shards(records, prefix, n_shards=8):
    N = len(records)
    per = math.ceil(N / n_shards)
    for k in range(n_shards):
        s = k*per
        e = min((k+1)*per, N)
        chunk = records[s:e]
        np.save(os.path.join(OUT_ROOT, f"{prefix}_chunk{k}.npy"), np.array(chunk, dtype=object), allow_pickle=True)
        # lightweight metadata
        meta = [{"planet_id": r["planet_id"], "visit": r["visit"]} for r in chunk]
        with open(os.path.join(OUT_ROOT, f"{prefix}_chunk{k}.json"), "w") as f:
            json.dump(meta, f)


# ------- run processing -------
print("Processing training data...")
train_airs, train_fgs = process_dataset(TRAIN_ROOT, "train")
save_shards(train_airs, "train_airs", 8)
save_shards(train_fgs,  "train_fgs", 8)

print("Processing test data...")
test_airs, test_fgs = process_dataset(TEST_ROOT, "test")
save_shards(test_airs, "test_airs", 1)
save_shards(test_fgs,  "test_fgs", 1)

print("Done. Processed files saved to:", OUT_ROOT)


Found 11250 valid AIRS timestamps, 135000 valid FGS timestamps
Generated 5625 AIRS CDS times, 67500 FGS CDS times
Processing test data...
Found 1 planets under ./data\test


Processing test: 100%|██████████| 1/1 [00:07<00:00,  7.96s/it]

Done. Processed files saved to: ./processed
